# 03 — Optimization Strategies

Compare all portfolio construction methods:
1. Minimum Variance
2. Maximum Sharpe Ratio (Tangency Portfolio)
3. Maximum Utility (with investor's risk aversion A)
4. Risk Parity (Equal Risk Contribution)
5. CVaR Minimization
6. Efficient Frontier visualization
7. Out-of-sample backtest comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.estimators.returns import historical_mean
from src.estimators.covariance import ledoit_wolf_covariance
from src.optimizers.markowitz import minimum_variance, maximum_sharpe, maximum_utility, efficient_frontier
from src.optimizers.risk_parity import risk_parity, risk_contribution_report
from src.optimizers.cvar_optimizer import cvar_optimization, portfolio_cvar
from src.risk.metrics import portfolio_report
from src.utils.plotting import plot_efficient_frontier, plot_weights_bar, plot_risk_contributions, plot_cumulative_returns
from config import *

returns = pd.read_csv('../data/returns.csv', index_col=0, parse_dates=True)
cov = pd.read_csv('../data/cov_ledoit_wolf.csv', index_col=0)
mu  = historical_mean(returns)

print(f'Assets: {list(returns.columns)}')
print(f'Risk-free rate: {RISK_FREE_RATE_ANNUAL*100:.1f}%')

## 1. Compute All Portfolios

In [ ]:
p_minvar  = minimum_variance(mu, cov, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_sharpe  = maximum_sharpe(mu, cov, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_util    = maximum_utility(mu, cov, RISK_AVERSION, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_rp      = risk_parity(mu, cov, RISK_FREE_RATE_ANNUAL)
p_cvar    = cvar_optimization(returns, mu, cov, CVAR_ALPHA, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)

portfolios = {
    'Min Variance':  p_minvar,
    'Max Sharpe':    p_sharpe,
    f'Max Utility (A={RISK_AVERSION})': p_util,
    'Risk Parity':   p_rp,
    f'Min CVaR ({int((1-CVAR_ALPHA)*100)}%)': p_cvar,
}

for name, p in portfolios.items():
    print(p)

## 2. Efficient Frontier

In [ ]:
print('Computing efficient frontier (this may take ~30 seconds)...')
frontier = efficient_frontier(mu, cov, RISK_FREE_RATE_ANNUAL, N_FRONTIER_POINTS, MIN_WEIGHT, MAX_WEIGHT)
print(f'Computed {len(frontier)} frontier portfolios')

fig = plot_efficient_frontier(frontier, portfolios, RISK_FREE_RATE_ANNUAL,
                               save_path='../results/03_frontier.png')
plt.show()

## 3. Weights Comparison

In [ ]:
fig = plot_weights_bar(portfolios, save_path='../results/03_weights.png')
plt.show()

# Detailed weights table
weights_df = pd.DataFrame({name: p.weights for name, p in portfolios.items()}) * 100
print('\nWeights (%):')
display(weights_df.round(1))

## 4. Risk Contribution Analysis

In [ ]:
# Risk Parity should show equal contributions
rc_rp    = risk_contribution_report(p_rp.weights, cov)
rc_sharpe = risk_contribution_report(p_sharpe.weights, cov)

print('Risk Parity — Risk Contributions (should be ~equal):')
display(rc_rp)

print('\nMax Sharpe — Risk Contributions (typically concentrated):')
display(rc_sharpe)

fig = plot_risk_contributions(rc_rp, title='Risk Parity Portfolio — Risk Budget',
                               save_path='../results/03_risk_contributions.png')
plt.show()

## 5. CVaR Analysis

In [ ]:
print('CVaR Analysis (Historical Simulation):\n')
for name, p in portfolios.items():
    cvar_stats = portfolio_cvar(p.weights, returns, alpha=CVAR_ALPHA)
    print(f'{name}:')
    print(f'  Monthly VaR 95%:  {cvar_stats["VaR (monthly)"]*100:.2f}%')
    print(f'  Monthly CVaR 95%: {cvar_stats["CVaR (monthly)"]*100:.2f}%')
    print(f'  Annual CVaR:      {cvar_stats["CVaR (annual)"]*100:.2f}%')
    print(f'  Worst month:      {cvar_stats["Worst return"]*100:.2f}%\n')

## 6. Cumulative Return Backtest

In [ ]:
weights_for_plot = {name: p.weights for name, p in portfolios.items()}
fig = plot_cumulative_returns(returns, weights_for_plot, RISK_FREE_RATE_MONTHLY,
                               save_path='../results/03_cumulative.png')
plt.show()

## 7. Full Risk Report

In [ ]:
print('Full Risk/Performance Report:\n')
report_rows = []
for name, p in portfolios.items():
    r = portfolio_report(p.weights, returns, mu, cov, RISK_FREE_RATE_ANNUAL, RISK_AVERSION)
    r['Strategy'] = name
    report_rows.append(r)

report_df = pd.DataFrame(report_rows).set_index('Strategy')
display(report_df.T)